<a href="https://colab.research.google.com/github/Vadapao26/WiseWaste-Production-Performance-Analytics/blob/main/Production_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# *WiseWaste-Production-Performance-Analytics*

In [ ]:
from google.colab import files

uploaded = files.upload()

import os
import re
import unicodedata
import pandas as pd
import numpy as np
from difflib import get_close_matches
from google.colab import files

# =========================================================
# 1. FILE DETECTION
# =========================================================
if 'uploaded' in globals() and isinstance(uploaded, dict) and len(uploaded) > 0:
    uploaded_keys = list(uploaded.keys())
    csv_candidates = [f for f in uploaded_keys if f.lower().endswith('.csv')]
    if len(csv_candidates) == 0:
        raise FileNotFoundError("No CSV uploaded. Please upload Production_cleaned.csv")
    INPUT_FILE = csv_candidates[0]
else:
    csv_candidates = [f for f in os.listdir('.') if f.lower().endswith('.csv')]
    if len(csv_candidates) == 0:
        raise FileNotFoundError("No CSV found in the working directory.")
    preferred = [f for f in csv_candidates if 'production_cleaned' in f.lower()]
    INPUT_FILE = preferred[0] if preferred else csv_candidates[0]

print("Using source file:", INPUT_FILE)

# =========================================================
# 2. LOAD SOURCE DATA
# =========================================================
df = pd.read_csv(INPUT_FILE)
df.columns = [c.strip() for c in df.columns]

required_columns = [
    'Production Code', 'Date', 'Shift', 'Process-Equipment',
    'Time Taken in Hrs', 'No. of Staff Present',
    'Material', 'Quantity in Kg', 'Total Rejection in Kg',
    'Materials', 'Material From', 'Material Quantity'
]

missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# =========================================================
# 3. CLEAN / STANDARDISE
# =========================================================
text_cols = ['Production Code', 'Date', 'Shift', 'Process-Equipment', 'Material', 'Materials', 'Material From']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

num_cols = ['Time Taken in Hrs', 'No. of Staff Present', 'Quantity in Kg', 'Total Rejection in Kg', 'Material Quantity']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

# =========================================================
# 4. PROCESS / EQUIPMENT SPLIT
# =========================================================
parts = df['Process-Equipment'].astype(str).str.split('-', n=1, expand=True)
df['Process'] = parts[0].str.strip().str.title()
df['Equipment'] = parts[1].str.strip()

# =========================================================
# 5. MATERIAL CATEGORY MAPPING
#    (exact + fuzzy + rule-based fallback)
# =========================================================
category_keywords = {
    'Paper': [
        'paper_', 'paper ', 'carton', 'tissue', 'paper cup', 'color paper'
    ],
    'Metal': [
        'metal_', 'aluminum', 'aluminium', 'stainless steel', 'copper', 'mild steel', 'iron'
    ],
    'Glass': [
        'glass_', 'glass bottle', 'broken glass', 'windshield'
    ],
    'Plastics': [
        'plastics_', 'plastic_', 'pete', 'hdpe', 'ldpe', 'pp_', 'pp ', 'pe_', 'pe ',
        'pvc', 'ps_', 'thermocol', 'foam', 'mlp', 'tetrapak', 'tetra pak',
        'bale straps', 'mixed rigid plastics', 'mixed flexible plastics',
        'lvp rigid plastic', 'lvp flexible plastics'
    ],
    'Textile Waste': [
        'textile waste', 'textile'
    ],
    'E-Waste': [
        'e-waste', 'ewaste', 'e waste'
    ],
    'Wet Waste': [
        'wet waste', 'garden waste', 'compost', 'food waste'
    ],
    'Sanitary Waste': [
        'sanitary waste'
    ],
    'C & D Waste': [
        'c&d_', 'c&d ', 'c & d', 'pop', 'gypsum boards', 'ceramics',
        'false ceiling', 'tiles', 'mixed debris', 'concrete'
    ],
    'Others': [
        'coconut shell', 'tender coconut', 'footwear', 'wood', 'rubber',
        'others', 'unsorted drywaste', 'drywaste_unsorted', 'drywaste',
        'reject', 'lvp (co-processing)', 'co-processing', 'lvp'
    ]
}

# Exact map for common source values
exact_material_category = {
    'Paper_Color Paper': 'Paper',
    'Paper_Carton Box': 'Paper',
    'Metal_Aluminum_Can': 'Metal',
    'Metal_Aluminum_Foil': 'Metal',
    'Glass_Glass Bottle': 'Glass',
    'Mixed Rigid Plastics': 'Plastics',
    'Mixed Flexible Plastics': 'Plastics',
    'Textile Waste': 'Textile Waste',
    'E-waste': 'E-Waste',
    'Coconut Shell': 'Others',
    'Footwear': 'Others',
    'Rejects': 'Others',
    'LVP (Co-Processing)': 'Others',
    'Unsorted Drywaste': 'Others',
    'Water Rejects-PET': 'Others'
}

def normalise_text(x):
    if pd.isna(x):
        return ''
    x = str(x).strip().lower()
    x = unicodedata.normalize('NFKD', x)
    x = x.replace('&', ' and ')
    x = x.replace('-', ' ')
    x = x.replace('_', ' ')
    x = re.sub(r'\s+', ' ', x)
    return x.strip()

norm_lookup = {normalise_text(k): v for k, v in exact_material_category.items()}

def map_category(material_name):
    if pd.isna(material_name):
        return 'Unmapped'
    raw = str(material_name).strip()

    if raw in exact_material_category:
        return exact_material_category[raw]

    norm = normalise_text(raw)

    if norm in norm_lookup:
        return norm_lookup[norm]

    fuzzy = get_close_matches(norm, list(norm_lookup.keys()), n=1, cutoff=0.88)
    if fuzzy:
        return norm_lookup[fuzzy[0]]

    for category, keywords in category_keywords.items():
        for kw in keywords:
            if kw in norm:
                return category

    return 'Unmapped'

# =========================================================
# 6. RUN-LEVEL TABLE (Approach A)
#    Staff/time counted once per run
# =========================================================
run_key = ['Production Code', 'Date', 'Shift', 'Process', 'Equipment', 'Process-Equipment']

base_runs = (
    df.groupby(run_key, dropna=False)
      .agg({
          'Time Taken in Hrs': 'max',
          'No. of Staff Present': 'max',
          'Quantity in Kg': 'max',
          'Total Rejection in Kg': 'max'
      })
      .reset_index()
)

material_totals_per_run = (
    df.groupby(run_key, dropna=False)['Material Quantity']
      .sum(min_count=1)
      .reset_index()
      .rename(columns={'Material Quantity': 'MaterialQty_Sum'})
)

run_df = base_runs.merge(material_totals_per_run, on=run_key, how='left')

# Agreed quantity logic
run_df['Production_Qty_kg'] = np.where(
    run_df['Process'] == 'Sorting',
    run_df['Quantity in Kg'],
    run_df['MaterialQty_Sum']
)

run_df['Staff_Hours'] = run_df['No. of Staff Present'] * run_df['Time Taken in Hrs']
run_df['Kg_per_Person'] = run_df['Production_Qty_kg'] / run_df['No. of Staff Present']
run_df['Kg_per_Hour'] = run_df['Production_Qty_kg'] / run_df['Time Taken in Hrs']
run_df['Kg_per_Staff_Hour'] = run_df['Production_Qty_kg'] / run_df['Staff_Hours']
run_df['Explicit_Rejection_kg'] = run_df['Total Rejection in Kg'].fillna(0)
run_df['Explicit_Rejection_Pct'] = np.where(
    run_df['Production_Qty_kg'] > 0,
    100 * run_df['Explicit_Rejection_kg'] / run_df['Production_Qty_kg'],
    np.nan
)

# =========================================================
# 7. MATERIAL-LONG TABLE
#    FIXED BUG: we consistently rename 'Material From' -> 'Material_From'
# =========================================================

# Sorting rows
sorting_material = (
    df[df['Process'] == 'Sorting'][[
        'Production Code', 'Date', 'Shift', 'Process', 'Equipment',
        'Material', 'Quantity in Kg'
    ]]
    .copy()
    .rename(columns={
        'Material': 'Material_Name',
        'Quantity in Kg': 'Qty_kg'
    })
)
sorting_material['Material_From'] = sorting_material['Material_Name']

# Bagging / Bailing rows
other_material = (
    df[df['Process'].isin(['Bagging', 'Bailing'])][[
        'Production Code', 'Date', 'Shift', 'Process', 'Equipment',
        'Materials', 'Material From', 'Material Quantity'
    ]]
    .copy()
    .rename(columns={
        'Materials': 'Material_Name',
        'Material From': 'Material_From',      # ✅ FIX
        'Material Quantity': 'Qty_kg'
    })
)

# Combined long table
material_long = pd.concat([
    sorting_material[['Production Code', 'Date', 'Shift', 'Process', 'Equipment', 'Material_Name', 'Material_From', 'Qty_kg']],
    other_material[['Production Code', 'Date', 'Shift', 'Process', 'Equipment', 'Material_Name', 'Material_From', 'Qty_kg']]
], ignore_index=True)

material_long['Material_Category'] = material_long['Material_Name'].apply(map_category)
material_long['Source_Category'] = material_long['Material_From'].apply(map_category)

# =========================================================
# 8. HELPER AGGREGATION
# =========================================================
def summarise_runs(group_cols):
    out = (
        run_df.groupby(group_cols, dropna=False)
              .agg(
                  Total_Kg=('Production_Qty_kg', 'sum'),
                  Total_Staff=('No. of Staff Present', 'sum'),
                  Total_Hours=('Time Taken in Hrs', 'sum'),
                  Total_Staff_Hours=('Staff_Hours', 'sum'),
                  Runs=('Production Code', 'nunique'),
                  Explicit_Rejection_kg=('Explicit_Rejection_kg', 'sum')
              )
              .reset_index()
    )

    out['Kg_per_Person'] = out['Total_Kg'] / out['Total_Staff']
    out['Kg_per_Hour'] = out['Total_Kg'] / out['Total_Hours']
    out['Kg_per_Staff_Hour'] = out['Total_Kg'] / out['Total_Staff_Hours']
    out['Explicit_Rejection_Pct'] = np.where(
        out['Total_Kg'] > 0,
        100 * out['Explicit_Rejection_kg'] / out['Total_Kg'],
        np.nan
    )
    return out

# Reject rows as output material
reject_rows = material_long[
    material_long['Material_Name'].fillna('').str.strip().str.lower() == 'rejects'
].copy()

reject_by_pe = (
    reject_rows.groupby(['Date', 'Shift', 'Process', 'Equipment'], dropna=False)['Qty_kg']
               .sum()
               .reset_index()
               .rename(columns={'Qty_kg': 'Rejects_as_Material_kg'})
)

reject_by_process = (
    reject_rows.groupby(['Process'], dropna=False)['Qty_kg']
               .sum()
               .reset_index()
               .rename(columns={'Qty_kg': 'Rejects_as_Material_kg'})
)

# =========================================================
# 9. OVERALL ANALYSIS
# =========================================================
overall_process = summarise_runs(['Process']).sort_values('Total_Kg', ascending=False)
overall_process = overall_process.merge(reject_by_process, on='Process', how='left')
overall_process['Rejects_as_Material_kg'] = overall_process['Rejects_as_Material_kg'].fillna(0)
overall_process['Rejects_as_Material_Pct'] = np.where(
    overall_process['Total_Kg'] > 0,
    100 * overall_process['Rejects_as_Material_kg'] / overall_process['Total_Kg'],
    np.nan
)
overall_process['Pct_of_Total_Production'] = 100 * overall_process['Total_Kg'] / overall_process['Total_Kg'].sum()

overall_material = (
    material_long.groupby(['Material_Name', 'Material_Category'], dropna=False)['Qty_kg']
                .sum()
                .reset_index()
                .sort_values('Qty_kg', ascending=False)
)
overall_material['Pct_of_Total_Material'] = 100 * overall_material['Qty_kg'] / overall_material['Qty_kg'].sum()

overall_category = (
    material_long.groupby(['Material_Category'], dropna=False)['Qty_kg']
                .sum()
                .reset_index()
                .sort_values('Qty_kg', ascending=False)
)
overall_category['Pct_of_Total_Material'] = 100 * overall_category['Qty_kg'] / overall_category['Qty_kg'].sum()

# =========================================================
# 10. PROCESS & EQUIPMENT ANALYSIS
# =========================================================
process_equipment = summarise_runs(['Process', 'Equipment']).sort_values(['Process', 'Equipment'])

reject_pe_rollup = (
    reject_by_pe.groupby(['Process', 'Equipment'], dropna=False)['Rejects_as_Material_kg']
                .sum()
                .reset_index()
)

process_equipment = process_equipment.merge(
    reject_pe_rollup,
    on=['Process', 'Equipment'],
    how='left'
)
process_equipment['Rejects_as_Material_kg'] = process_equipment['Rejects_as_Material_kg'].fillna(0)
process_equipment['Rejects_as_Material_Pct'] = np.where(
    process_equipment['Total_Kg'] > 0,
    100 * process_equipment['Rejects_as_Material_kg'] / process_equipment['Total_Kg'],
    np.nan
)

pe_material_mix = (
    material_long.groupby(['Process', 'Equipment', 'Material_Name', 'Material_Category'], dropna=False)['Qty_kg']
                .sum().reset_index()
)
pe_material_mix['Pct_within_Process_Equipment'] = (
    pe_material_mix.groupby(['Process', 'Equipment'])['Qty_kg']
                   .transform(lambda s: 100 * s / s.sum())
)
pe_material_mix = pe_material_mix.sort_values(['Process', 'Equipment', 'Qty_kg'], ascending=[True, True, False])

pe_category_mix = (
    material_long.groupby(['Process', 'Equipment', 'Material_Category'], dropna=False)['Qty_kg']
                .sum().reset_index()
)
pe_category_mix['Pct_within_Process_Equipment'] = (
    pe_category_mix.groupby(['Process', 'Equipment'])['Qty_kg']
                   .transform(lambda s: 100 * s / s.sum())
)
pe_category_mix = pe_category_mix.sort_values(['Process', 'Equipment', 'Qty_kg'], ascending=[True, True, False])

# =========================================================
# 11. EFFICIENCY ANALYSIS
# =========================================================
daily_process = summarise_runs(['Date', 'Process']).sort_values(['Date', 'Process'])
shift_process = summarise_runs(['Date', 'Shift', 'Process']).sort_values(['Date', 'Shift', 'Process'])
daily_equipment = summarise_runs(['Date', 'Process', 'Equipment']).sort_values(['Date', 'Process', 'Equipment'])
shift_equipment = summarise_runs(['Date', 'Shift', 'Process', 'Equipment']).sort_values(['Date', 'Shift', 'Process', 'Equipment'])

process_rankings = overall_process[['Process', 'Total_Kg', 'Kg_per_Person', 'Kg_per_Hour', 'Kg_per_Staff_Hour', 'Rejects_as_Material_Pct']].copy()
process_rankings['Rank_Kg_per_Person'] = process_rankings['Kg_per_Person'].rank(method='dense', ascending=False)
process_rankings['Rank_Kg_per_Hour'] = process_rankings['Kg_per_Hour'].rank(method='dense', ascending=False)
process_rankings['Rank_Kg_per_Staff_Hour'] = process_rankings['Kg_per_Staff_Hour'].rank(method='dense', ascending=False)
process_rankings['Rank_Lowest_Reject_Pct'] = process_rankings['Rejects_as_Material_Pct'].rank(method='dense', ascending=True)

equipment_rankings = process_equipment[['Process', 'Equipment', 'Total_Kg', 'Kg_per_Person', 'Kg_per_Hour', 'Kg_per_Staff_Hour', 'Rejects_as_Material_Pct']].copy()
equipment_rankings['Rank_Kg_per_Person_within_Process'] = equipment_rankings.groupby('Process')['Kg_per_Person'].rank(method='dense', ascending=False)
equipment_rankings['Rank_Kg_per_Hour_within_Process'] = equipment_rankings.groupby('Process')['Kg_per_Hour'].rank(method='dense', ascending=False)
equipment_rankings['Rank_Kg_per_Staff_Hour_within_Process'] = equipment_rankings.groupby('Process')['Kg_per_Staff_Hour'].rank(method='dense', ascending=False)
equipment_rankings['Rank_Lowest_Reject_within_Process'] = equipment_rankings.groupby('Process')['Rejects_as_Material_Pct'].rank(method='dense', ascending=True)

best_days = daily_process.sort_values(['Process', 'Kg_per_Person'], ascending=[True, False]).groupby('Process').head(5)
worst_days = daily_process.sort_values(['Process', 'Kg_per_Person'], ascending=[True, True]).groupby('Process').head(5)

best_runs = run_df.sort_values(['Process', 'Kg_per_Person'], ascending=[True, False]).groupby('Process').head(5)
worst_runs = run_df.sort_values(['Process', 'Kg_per_Person'], ascending=[True, True]).groupby('Process').head(5)

# =========================================================
# 12. MATERIAL CHARACTERISATION
# =========================================================
material_characterisation = (
    material_long.groupby(['Material_Name', 'Material_Category', 'Process'], dropna=False)['Qty_kg']
                .sum().reset_index()
)
material_characterisation['Pct_within_Process'] = (
    material_characterisation.groupby('Process')['Qty_kg']
                              .transform(lambda s: 100 * s / s.sum())
)
material_characterisation = material_characterisation.sort_values(['Process', 'Qty_kg'], ascending=[True, False])

category_by_process = (
    material_long.groupby(['Process', 'Material_Category'], dropna=False)['Qty_kg']
                .sum().reset_index()
)
category_by_process['Pct_within_Process'] = (
    category_by_process.groupby('Process')['Qty_kg']
                       .transform(lambda s: 100 * s / s.sum())
)
category_by_process = category_by_process.sort_values(['Process', 'Qty_kg'], ascending=[True, False])

mapping_review = (
    material_long.groupby(['Material_Name', 'Material_Category'], dropna=False)['Qty_kg']
                .sum().reset_index()
                .sort_values('Qty_kg', ascending=False)
)
mapping_review['Mapping_Status'] = np.where(
    mapping_review['Material_Category'] == 'Unmapped',
    'Review Needed',
    'Mapped'
)

# =========================================================
# 13. SOURCE-TO-OUTPUT FLOW
# =========================================================
source_to_output = (
    material_long.groupby(['Process', 'Material_From', 'Source_Category', 'Material_Name', 'Material_Category'], dropna=False)['Qty_kg']
                .sum().reset_index()
)
source_to_output['Pct_within_Source_Process'] = (
    source_to_output.groupby(['Process', 'Material_From'])['Qty_kg']
                    .transform(lambda s: 100 * s / s.sum())
)
source_to_output = source_to_output.sort_values(['Process', 'Material_From', 'Qty_kg'], ascending=[True, True, False])

source_cat_to_output_cat = (
    material_long.groupby(['Process', 'Source_Category', 'Material_Category'], dropna=False)['Qty_kg']
                .sum().reset_index()
)
source_cat_to_output_cat['Pct_within_Source_Category_Process'] = (
    source_cat_to_output_cat.groupby(['Process', 'Source_Category'])['Qty_kg']
                            .transform(lambda s: 100 * s / s.sum())
)
source_cat_to_output_cat = source_cat_to_output_cat.sort_values(['Process', 'Source_Category', 'Qty_kg'], ascending=[True, True, False])

# =========================================================
# 14. REJECTION & LOSS ANALYSIS
# =========================================================
reject_by_process_sheet = overall_process[['Process', 'Total_Kg', 'Explicit_Rejection_kg', 'Explicit_Rejection_Pct', 'Rejects_as_Material_kg', 'Rejects_as_Material_Pct']].copy()
reject_by_equipment_sheet = process_equipment[['Process', 'Equipment', 'Total_Kg', 'Explicit_Rejection_kg', 'Explicit_Rejection_Pct', 'Rejects_as_Material_kg', 'Rejects_as_Material_Pct']].copy()
reject_by_day_sheet = daily_process[['Date', 'Process', 'Total_Kg', 'Explicit_Rejection_kg', 'Explicit_Rejection_Pct']].copy()

reject_by_source = (
    reject_rows.groupby(['Process', 'Material_From', 'Source_Category'], dropna=False)['Qty_kg']
               .sum().reset_index()
               .rename(columns={'Qty_kg': 'Rejects_kg'})
)

reject_by_category = (
    reject_rows.groupby(['Process', 'Source_Category'], dropna=False)['Qty_kg']
               .sum().reset_index()
               .rename(columns={'Qty_kg': 'Rejects_kg'})
)

# =========================================================
# 15. PLANNING NORMS / CAPACITY REFERENCE
# =========================================================
planning_process = overall_process[['Process', 'Total_Kg', 'Total_Staff', 'Total_Hours', 'Total_Staff_Hours', 'Runs', 'Kg_per_Person', 'Kg_per_Hour', 'Kg_per_Staff_Hour']].copy()
planning_process['Avg_Kg_per_Run'] = planning_process['Total_Kg'] / planning_process['Runs']
planning_process['Staff_per_1000Kg'] = np.where(planning_process['Total_Kg'] > 0, 1000 * planning_process['Total_Staff'] / planning_process['Total_Kg'], np.nan)
planning_process['Hours_per_1000Kg'] = np.where(planning_process['Total_Kg'] > 0, 1000 * planning_process['Total_Hours'] / planning_process['Total_Kg'], np.nan)
planning_process['StaffHours_per_1000Kg'] = np.where(planning_process['Total_Kg'] > 0, 1000 * planning_process['Total_Staff_Hours'] / planning_process['Total_Kg'], np.nan)

planning_equipment = process_equipment[['Process', 'Equipment', 'Total_Kg', 'Total_Staff', 'Total_Hours', 'Total_Staff_Hours', 'Runs', 'Kg_per_Person', 'Kg_per_Hour', 'Kg_per_Staff_Hour']].copy()
planning_equipment['Avg_Kg_per_Run'] = planning_equipment['Total_Kg'] / planning_equipment['Runs']
planning_equipment['Staff_per_1000Kg'] = np.where(planning_equipment['Total_Kg'] > 0, 1000 * planning_equipment['Total_Staff'] / planning_equipment['Total_Kg'], np.nan)
planning_equipment['Hours_per_1000Kg'] = np.where(planning_equipment['Total_Kg'] > 0, 1000 * planning_equipment['Total_Hours'] / planning_equipment['Total_Kg'], np.nan)
planning_equipment['StaffHours_per_1000Kg'] = np.where(planning_equipment['Total_Kg'] > 0, 1000 * planning_equipment['Total_Staff_Hours'] / planning_equipment['Total_Kg'], np.nan)

run_outliers = run_df[['Production Code', 'Date', 'Shift', 'Process', 'Equipment', 'Production_Qty_kg', 'No. of Staff Present', 'Time Taken in Hrs', 'Kg_per_Person', 'Kg_per_Hour', 'Kg_per_Staff_Hour']].copy()
run_outliers['Process_Median_Kg_per_Person'] = run_outliers.groupby('Process')['Kg_per_Person'].transform('median')
run_outliers['Process_Median_Kg_per_Staff_Hour'] = run_outliers.groupby('Process')['Kg_per_Staff_Hour'].transform('median')
run_outliers['Low_Labour_Productivity_Flag'] = np.where(
    run_outliers['Kg_per_Person'] < 0.75 * run_outliers['Process_Median_Kg_per_Person'],
    'Low',
    ''
)
run_outliers['Low_Time_Productivity_Flag'] = np.where(
    run_outliers['Kg_per_Staff_Hour'] < 0.75 * run_outliers['Process_Median_Kg_per_Staff_Hour'],
    'Low',
    ''
)

# =========================================================
# 16. DASHBOARD (table-only, manager-facing)
# =========================================================
headline_kpis = pd.DataFrame({
    'Metric': [
        'Total Production (Kg)',
        'Total Runs',
        'Total Staff',
        'Total Hours',
        'Total Staff-Hours',
        'Facility Kg per Person',
        'Facility Kg per Hour',
        'Facility Kg per Staff-Hour',
        'Best Process by Kg per Person',
        'Best Equipment by Kg per Staff-Hour',
        'Top Material Category',
        'Total Explicit Rejection (Kg)',
        'Total Rejects as Material (Kg)'
    ],
    'Value': [
        round(run_df['Production_Qty_kg'].sum(), 2),
        int(run_df['Production Code'].nunique()),
        round(run_df['No. of Staff Present'].sum(), 2),
        round(run_df['Time Taken in Hrs'].sum(), 2),
        round(run_df['Staff_Hours'].sum(), 2),
        round(run_df['Production_Qty_kg'].sum() / run_df['No. of Staff Present'].sum(), 2) if run_df['No. of Staff Present'].sum() else np.nan,
        round(run_df['Production_Qty_kg'].sum() / run_df['Time Taken in Hrs'].sum(), 2) if run_df['Time Taken in Hrs'].sum() else np.nan,
        round(run_df['Production_Qty_kg'].sum() / run_df['Staff_Hours'].sum(), 2) if run_df['Staff_Hours'].sum() else np.nan,
        overall_process.sort_values('Kg_per_Person', ascending=False).iloc[0]['Process'] if len(overall_process) else np.nan,
        process_equipment.sort_values('Kg_per_Staff_Hour', ascending=False).iloc[0]['Equipment'] if len(process_equipment) else np.nan,
        overall_category.sort_values('Qty_kg', ascending=False).iloc[0]['Material_Category'] if len(overall_category) else np.nan,
        round(run_df['Explicit_Rejection_kg'].sum(), 2),
        round(reject_rows['Qty_kg'].sum(), 2) if len(reject_rows) else 0
    ]
})

manager_best_processes = overall_process[['Process', 'Total_Kg', 'Kg_per_Person', 'Kg_per_Hour', 'Kg_per_Staff_Hour', 'Rejects_as_Material_Pct']].sort_values('Kg_per_Staff_Hour', ascending=False).head(5)
manager_worst_processes = overall_process[['Process', 'Total_Kg', 'Kg_per_Person', 'Kg_per_Hour', 'Kg_per_Staff_Hour', 'Rejects_as_Material_Pct']].sort_values('Kg_per_Staff_Hour', ascending=True).head(5)
manager_best_equipment = process_equipment[['Process', 'Equipment', 'Total_Kg', 'Kg_per_Person', 'Kg_per_Hour', 'Kg_per_Staff_Hour', 'Rejects_as_Material_Pct']].sort_values('Kg_per_Staff_Hour', ascending=False).head(10)
manager_worst_equipment = process_equipment[['Process', 'Equipment', 'Total_Kg', 'Kg_per_Person', 'Kg_per_Hour', 'Kg_per_Staff_Hour', 'Rejects_as_Material_Pct']].sort_values('Kg_per_Staff_Hour', ascending=True).head(10)
manager_top_categories = overall_category.head(10)
manager_low_productivity_runs = run_outliers[
    (run_outliers['Low_Labour_Productivity_Flag'] == 'Low') |
    (run_outliers['Low_Time_Productivity_Flag'] == 'Low')
].head(20)

# =========================================================
# 17. FORMATTING
# =========================================================
all_frames = [
    df, run_df, overall_process, overall_material, overall_category,
    process_equipment, pe_material_mix, pe_category_mix,
    daily_process, shift_process, daily_equipment, shift_equipment,
    process_rankings, equipment_rankings, best_days, worst_days,
    best_runs, worst_runs, material_characterisation, category_by_process,
    mapping_review, source_to_output, source_cat_to_output_cat,
    reject_by_process_sheet, reject_by_equipment_sheet, reject_by_day_sheet,
    reject_by_source, reject_by_category, planning_process, planning_equipment,
    run_outliers, headline_kpis, manager_best_processes, manager_worst_processes,
    manager_best_equipment, manager_worst_equipment, manager_top_categories,
    manager_low_productivity_runs
]

for frame in all_frames:
    if 'Date' in frame.columns:
        frame['Date'] = pd.to_datetime(frame['Date'], errors='coerce').dt.strftime('%d %b %Y')
    numerics = frame.select_dtypes(include=[np.number]).columns
    frame[numerics] = frame[numerics].round(2)

# =========================================================
# 18. WRITE WORKBOOK
# =========================================================
OUTPUT_FILE = "Production_Facility_Analytics_Workbook.xlsx"

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    # Dashboard with multiple sections
    dashboard_sheet = 'Dashboard'
    headline_kpis.to_excel(writer, sheet_name=dashboard_sheet, index=False, startrow=0)
    manager_best_processes.to_excel(writer, sheet_name=dashboard_sheet, index=False, startrow=18)
    manager_worst_processes.to_excel(writer, sheet_name=dashboard_sheet, index=False, startrow=26)
    manager_best_equipment.to_excel(writer, sheet_name=dashboard_sheet, index=False, startrow=34)
    manager_worst_equipment.to_excel(writer, sheet_name=dashboard_sheet, index=False, startrow=48)
    manager_top_categories.to_excel(writer, sheet_name=dashboard_sheet, index=False, startrow=62)
    manager_low_productivity_runs.to_excel(writer, sheet_name=dashboard_sheet, index=False, startrow=76)

    # Analysis sheets
    overall_process.to_excel(writer, sheet_name='Overall Analysis', index=False)
    overall_material.to_excel(writer, sheet_name='Overall Materials', index=False)
    overall_category.to_excel(writer, sheet_name='Overall Categories', index=False)

    process_equipment.to_excel(writer, sheet_name='Process_Equipment_Analysis', index=False)
    pe_material_mix.to_excel(writer, sheet_name='PE_Material_Mix', index=False)
    pe_category_mix.to_excel(writer, sheet_name='PE_Category_Mix', index=False)

    daily_process.to_excel(writer, sheet_name='Efficiency_Daily_Process', index=False)
    shift_process.to_excel(writer, sheet_name='Efficiency_Shift_Process', index=False)
    daily_equipment.to_excel(writer, sheet_name='Efficiency_Daily_Equipment', index=False)
    shift_equipment.to_excel(writer, sheet_name='Efficiency_Shift_Equipment', index=False)

    process_rankings.to_excel(writer, sheet_name='Process_Rankings', index=False)
    equipment_rankings.to_excel(writer, sheet_name='Equipment_Rankings', index=False)
    best_days.to_excel(writer, sheet_name='Best_Days', index=False)
    worst_days.to_excel(writer, sheet_name='Worst_Days', index=False)
    best_runs.to_excel(writer, sheet_name='Best_Runs', index=False)
    worst_runs.to_excel(writer, sheet_name='Worst_Runs', index=False)

    material_characterisation.to_excel(writer, sheet_name='Material_Characterisation', index=False)
    category_by_process.to_excel(writer, sheet_name='Category_by_Process', index=False)
    mapping_review.to_excel(writer, sheet_name='Material_Mapping_Review', index=False)

    source_to_output.to_excel(writer, sheet_name='Source_to_Output_Flow', index=False)
    source_cat_to_output_cat.to_excel(writer, sheet_name='SourceCat_to_OutputCat', index=False)

    reject_by_process_sheet.to_excel(writer, sheet_name='Reject_by_Process', index=False)
    reject_by_equipment_sheet.to_excel(writer, sheet_name='Reject_by_Equipment', index=False)
    reject_by_day_sheet.to_excel(writer, sheet_name='Reject_by_Day', index=False)
    reject_by_source.to_excel(writer, sheet_name='Reject_by_Source', index=False)
    reject_by_category.to_excel(writer, sheet_name='Reject_by_Category', index=False)

    planning_process.to_excel(writer, sheet_name='Planning_Process_Norms', index=False)
    planning_equipment.to_excel(writer, sheet_name='Planning_Equipment_Norms', index=False)
    run_outliers.to_excel(writer, sheet_name='Run_Outliers', index=False)

    df.to_excel(writer, sheet_name='Source_Data', index=False)
    run_df.to_excel(writer, sheet_name='Run_Level_Data', index=False)

print(f"✅ Workbook created: {OUTPUT_FILE}")

# =========================================================
# 19. OPTIONAL DOWNLOAD
# =========================================================
files.download(OUTPUT_FILE)

Saving Production_cleaned.csv to Production_cleaned (1).csv
Using source file: Production_cleaned (1).csv
✅ Workbook created: Production_Facility_Analytics_Workbook.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>